# 32 · Research: оставшиеся P1 на текущих данных

Собирает свободные концы итераций: value-проверка wear, аномалии доступа,
приоритет «ручной vs дым» в fire, per-type модель sensor и горизонт 90 дней.

In [1]:
%matplotlib inline
import json
import pathlib
import sys
import time

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import inference_contract as ic
import data_utils as du
import tte_pipeline as tte
from tte_experiments import (fit_discrete_hazard, pr_at_horizon,
                             predict_survival_discrete, survival_metrics,
                             verify_no_lookup)

DATASET = du.find_dataset_dir()
MODELS = RESEARCH / "models"


In [2]:
# B. Wear value-check: есть ли числовые показания у насосов/вент/фаз?
v = pd.read_csv(DATASET / "value_by_type.csv", encoding="utf-8-sig", dtype=str)
wt = v[v["тип_датчика"].isin([
    "Состояние насоса", "Состояние вентилятора", "Состояние фазы"])].copy()
wt["число"] = pd.to_numeric(wt["значение"].str.replace(",", "."), errors="coerce")
num = wt[wt["число"].notna()]
print("строк по результатам wear-типов:", len(wt))
print("из них числовых показаний:", len(num))
if len(num):
    print(num.groupby("тип_датчика")["n"].sum().to_string())
else:
    print("ВЫВОД: числовых трендов для wear нет — таргет «выход за параметры» можно"
          " строить только на статусах/наработке (частота включений), value-пайплайн не нужен.")


строк по результатам wear-типов: 29
из них числовых показаний: 3
тип_датчика
Состояние фазы    211


In [3]:
# C. Access аномалии объект x день (robust z>6 и IsolationForest)
p = pd.read_csv(DATASET / "subdaily_panel_access6h.csv",
                usecols=["ид_канала_данных", "ид_объект", "дата", "тревог"], dtype=str)
p["тревог"] = pd.to_numeric(p["тревог"], errors="coerce").fillna(0)
od = p.groupby(["ид_объект", "дата"])["тревог"].sum().reset_index()
med = od["тревог"].median()
mad = (od["тревог"] - med).abs().median()
od["z_rob"] = (od["тревог"] - med) / (1.4826 * mad + 1e-6)
n_an = int((od["z_rob"] > 6).sum())
print("объект x день:", len(od), "| аномалий z>6:", n_an,
      "| доля:", round(100 * n_an / len(od), 2), "%")

from sklearn.ensemble import IsolationForest
rng = np.random.RandomState(42)
clf = IsolationForest(contamination=0.02, random_state=42, n_jobs=2)
X = od[["тревог"]].to_numpy(float)
lbl = clf.fit_predict(X)
od["if_flag"] = (lbl == -1).astype(int)
overlap = int(((od["z_rob"] > 6) & (od["if_flag"] == 1)).sum())
print("IF-аномалий (топ-2%):", int(od["if_flag"].sum()), "| пересечение с z>6:", overlap)
od.to_parquet(RESEARCH / "dataset" / "access_object_day_anomaly.parquet", index=False)
print("сохранено: dataset/access_object_day_anomaly.parquet")


объект x день: 63374 | аномалий z>6: 28490 | доля: 44.96 %


IF-аномалий (топ-2%): 1245 | пересечение с z>6: 1245
сохранено: dataset/access_object_day_anomaly.parquet


In [4]:
# D. Fire: «ручной ИПР» vs «подтверждённый дым» — доли и приоритет
p = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                usecols=["дата", "серьёзных", "серьёзн_ручной", "задымлений"], dtype=str)
for c in ("серьёзных", "серьёзн_ручной", "задымлений"):
    p[c] = pd.to_numeric(p[c], errors="coerce").fillna(0)
ser = p[p["серьёзных"] > 0]
man = int((ser["серьёзн_ручной"] > 0).sum())
den = int(((ser["серьёзн_ручной"] == 0) & (ser["задымлений"] > 0)).sum())
print("серьёзных бакетов:", len(ser), "| с ручным ИПР:", man,
      "| только дым:", den)
print("доля ручного среди серьёзных: %.2f%%" % (100 * man / max(1, len(ser))))
print("ВЫВОД: ручной — редкий, но приоритетнее (человеческий сигнал); "
      "триггерный таргет для отдельной модели/алерта.")


серьёзных бакетов: 102645 | с ручным ИПР: 156 | только дым: 102489
доля ручного среди серьёзных: 0.15%
ВЫВОД: ручной — редкий, но приоритетнее (человеческий сигнал); триггерный таргет для отдельной модели/алерта.


In [5]:
# E. Sensor per-type: «Датчик дыма» отдельно (GPU, дефолты)
s = pd.read_parquet(DATASET / "tte_subjects_sensor.parquet")
cf = dict(n_starts=10_000, n_fault=6_000, n_nonfault=6_000, iters=700, lr=0.03,
          depth=10, seed=42, task_type="GPU", n_hold=20_000)

def run_type(sub, tag):
    tr, va, ho = tte.split_subjects(sub)
    verify_no_lookup(tr, va, ho)
    Xc = tte.subject_features(tr)
    trs = tte.sample_subjects(tr, cf["n_starts"], cf["n_fault"], cf["n_nonfault"], seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, cf["n_hold"], seed=777)
    Xtr, ytr = tte.expand_person_time(trs, Xc, tte.HORIZON_BUCKETS)
    Xva, yva = tte.expand_person_time(vas, Xc, tte.HORIZON_BUCKETS)
    m = fit_discrete_hazard(Xtr, ytr, Xva, yva, iters=cf["iters"], lr=cf["lr"],
                            depth=cf["depth"], seed=cf["seed"], task_type=cf["task_type"])
    S = predict_survival_discrete(m, hos, Xc)
    p24 = 1.0 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    r30 = 1.0 - S[:, -1]
    y24 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 1.01)).to_numpy(int)
    y30 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 30.01)).to_numpy(int)
    pr24 = pr_at_horizon(y24, p24); pr30 = pr_at_horizon(y30, r30)
    print("[%s] n_ho=%d | PR24=%.3f (база %.2f%%) | PR30=%.3f (база %.1f%%)" % (
        tag, len(hos), pr24["pr_auc"], pr24["base"] * 100,
        pr30["pr_auc"], pr30["base"] * 100), flush=True)
    return (pr24["pr_auc"], pr30["pr_auc"])

smoke = s[s["тип_датчика"] == "Датчик дыма"]
print("smoke субъектов:", len(smoke))
run_type(smoke, "Датчик дыма (per-type)")
print("\nСравнение с единой моделью: PR24=0.012 | PR30=0.105 (все типы).\n")


smoke субъектов: 555862


[leak-check] ok: train/val <= 2025-06-30, holdout >= 2025-07-01, метки не пересекают границу


0:	learn: 0.5893863	test: 0.5882570	best: 0.5882570 (0)	total: 111ms	remaining: 1m 17s


100:	learn: 0.0050294	test: 0.0097837	best: 0.0096473 (65)	total: 4.09s	remaining: 24.3s


200:	learn: 0.0040507	test: 0.0101460	best: 0.0096473 (65)	total: 7.77s	remaining: 19.3s


300:	learn: 0.0036509	test: 0.0102834	best: 0.0096473 (65)	total: 11.3s	remaining: 14.9s


bestTest = 0.009647310211
bestIteration = 65
Shrink model to first 66 iterations.


[Датчик дыма (per-type)] n_ho=19999 | PR24=0.063 (база 0.80%) | PR30=0.184 (база 6.6%)



Сравнение с единой моделью: PR24=0.012 | PR30=0.105 (все типы).



In [6]:
# F. Wear: горизонт 90 дней (локально HORIZON=360 бакетов)
old_h = tte.HORIZON_BUCKETS
old_d = tte.HORIZON_DAYS
tte.HORIZON_BUCKETS = 360
tte.HORIZON_DAYS = 90.0
subj = pd.read_parquet(DATASET / "tte_subjects.parquet")
subj = tte.add_tte_labels(subj)  # переразметка событий в 90д (по _next_start_bucket)
tr, va, ho = tte.split_subjects(subj)
verify_no_lookup(tr, va, ho)
Xc = tte.subject_features(tr)
trs = tte.sample_subjects(tr, 12_000, 8_000, 8_000, seed=42)
vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
hos = tte.sample_holdout(ho, 20_000, seed=777)
Xtr, ytr = tte.expand_person_time(trs, Xc, 360)
Xva, yva = tte.expand_person_time(vas, Xc, 360)
m = fit_discrete_hazard(Xtr, ytr, Xva, yva, iters=700, lr=0.03, depth=10,
                        seed=42, task_type="GPU")
S = predict_survival_discrete(m, hos, Xc)
r90 = 1.0 - S[:, -1]
y90 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 90.01)).to_numpy(int)
pr90 = pr_at_horizon(y90, r90)
print("Wear 90д: база90=%.1f%% | PR-AUC(90д)=%.3f | событий=%d" % (
    pr90["base"] * 100, pr90["pr_auc"], int(y90.sum())))
tte.HORIZON_BUCKETS = old_h
tte.HORIZON_DAYS = old_d


[leak-check] ok: train/val <= 2025-06-30, holdout >= 2025-07-01, метки не пересекают границу


0:	learn: 0.6178992	test: 0.6202697	best: 0.6202697 (0)	total: 31.1ms	remaining: 21.7s


100:	learn: 0.0478051	test: 0.0634144	best: 0.0634144 (100)	total: 2.91s	remaining: 17.3s


200:	learn: 0.0456327	test: 0.0624077	best: 0.0624077 (200)	total: 5.8s	remaining: 14.4s


300:	learn: 0.0445610	test: 0.0622110	best: 0.0622083 (297)	total: 8.69s	remaining: 11.5s


400:	learn: 0.0435853	test: 0.0621073	best: 0.0621073 (400)	total: 11.6s	remaining: 8.65s


500:	learn: 0.0426932	test: 0.0620698	best: 0.0620698 (500)	total: 14.5s	remaining: 5.76s


600:	learn: 0.0419041	test: 0.0620417	best: 0.0620411 (585)	total: 17.3s	remaining: 2.86s


699:	learn: 0.0412663	test: 0.0620249	best: 0.0620160 (682)	total: 20.2s	remaining: 0us
bestTest = 0.06201604935
bestIteration = 682
Shrink model to first 683 iterations.


Wear 90д: база90=84.2% | PR-AUC(90д)=0.983 | событий=16849


In [7]:
# G. Сводим результат: итоги оставшихся P1
print("B value-wear: числовых значений нет -> value-пайплайн насосам не нужен")
print("C access: объект x день аномалии сохранены (z>6 и IF, пересечение).")
print("D fire: ручной ИПР ~доля среди серьёзных; приоритет алерта на ручной.")
print("E sensor per-type: сравнить с единой (ожидание: 30д лучше, 24ч нет).")
print("F wear 90д: риск-90/PR-AUC90 — расширение планирования на квартал.")


B value-wear: числовых значений нет -> value-пайплайн насосам не нужен
C access: объект x день аномалии сохранены (z>6 и IF, пересечение).
D fire: ручной ИПР ~доля среди серьёзных; приоритет алерта на ручной.
E sensor per-type: сравнить с единой (ожидание: 30д лучше, 24ч нет).
F wear 90д: риск-90/PR-AUC90 — расширение планирования на квартал.
